# Proyecto de Magíster: Detección de Anomalías y Posibles Fraudes en Permisos de Circulación

## Problemática
En la administración municipal, el cobro de los Permisos de Circulación Vehicular depende fuertemente de datos ingresados manualmente y tasaciones. Debido a la falta de validación estandarizada en los sistemas locales, existen múltiples errores de digitación, inconsistencias de formato y, potencialmente, **fraudes y evasiones** (por ejemplo, vehículos de alto valor comercial registrados con características alteradas para pagar menos, o modificaciones vehiculares no declaradas). Esta desestructuración de la información genera ineficiencias en la gestión pública y grandes pérdidas de ingresos municipales.

## Objetivo General
Desarrollar un pipeline de datos y un modelo de Machine Learning basado en **Aprendizaje No Supervisado (Detección de Anomalías)** para identificar patrones inusuales y registros atípicos en la base de datos de permisos de circulación, permitiendo alertar a la municipalidad sobre posibles fraudes o errores graves de tasación.

## Objetivos Específicos
1. Cargar y realizar una exploración inicial de los datos municipales (EDA).

In [ ]:
# Importación de librerías iniciales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización y advertencias
pd.set_option('display.max_columns', None) # Para asegurar que se vean todas las columnas al imprimir
import warnings
warnings.filterwarnings('ignore') # Limpiar la salida de advertencias rojas

print("Librerías iniciales importadas correctamente.")

Librerías iniciales importadas correctamente.


## 1. Carga de Datos y Exploración Inicial
En esta sección leemos el dataset en formato CSV, lo transformamos en un DataFrame (`df`) y visualizamos las primeras 10 filas para entender la estructura de nuestras columnas y el nivel de "suciedad" inicial de los datos.

In [ ]:
# Lectura del archivo CSV y exportación a DataFrame (df)
# Asegúrate de haber subido el archivo 'permiso-circulacion-2026.csv' a tu entorno
file_path = 'permiso-circulacion-2026.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Dataset cargado exitosamente. Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas.\n")

    # Mostrar las primeras 10 filas (y por ende, las columnas)
    display(df.head(10))

except FileNotFoundError:
    print(f"Error: No se encontró el archivo '{file_path}'.")
    print("Por favor, ve al panel izquierdo de Colab (ícono de carpeta) y sube el archivo CSV primero.")

Dataset cargado exitosamente. Dimensiones: 3195 filas y 21 columnas.



,_id,Fecha_Emision,Ano_Proceso,Metodo de Pago,Cuotas Permiso,Codigo_SII,Comuna_Propietario,Comuna_Permiso,Valor_Contado,Total_a_Pagar,Ano_Fabricacion,TipoVehiculo,Marca,Modelo,Cilindrada,Equipamiento,Combustible,Transmision,Tonelaje,Comuna_Anterior,Fecha_Vencimiento
0,1,2026-03-20T00:00:00,2026,PRESENCIAL,Total,SU188001078,AISEN,RIO IBANEZ,33715,37390,1978,AUTOMOVIL,PLYMOUTH,VOLARE,3700.0,Norm,Benc,Mec,0,RIO IBAÑEZ,2026-03-31T00:00:00
1,2,2026-03-20T00:00:00,2026,PRESENCIAL,Total,SU188001078,AISEN,RIO IBANEZ,34876,34876,1978,AUTOMOVIL,PLYMOUTH,VOLARE,3700.0,Norm,Benc,Mec,0,RIO IBAÑEZ,2027-03-31T00:00:00
2,3,2026-03-20T00:00:00,2026,PRESENCIAL,Total,SU188001078,AISEN,RIO IBANEZ,30885,43076,1978,AUTOMOVIL,PLYMOUTH,VOLARE,3700.0,Norm,Benc,Mec,0,RIO IBAÑEZ,2024-03-31T00:00:00
3,4,2026-03-20T00:00:00,2026,PRESENCIAL,Total,SU188001078,AISEN,RIO IBANEZ,32333,40626,1978,AUTOMOVIL,PLYMOUTH,VOLARE,3700.0,Norm,Benc,Mec,0,RIO IBAÑEZ,2025-03-31T00:00:00
4,5,2026-04-08T00:00:00,2026,PRESENCIAL,Total,NaN,COYHAIQUE,RIO IBANEZ,137846,145072,1988,CAMION,MERCEDES BENZ,L1114 48,NaN,NORM,DIES,MEC,8000,RIO BUENO,2026-09-30T00:00:00
5,6,2026-03-06T00:00:00,2026,PRESENCIAL,Total,MT139000113,RENCA,RIO IBANEZ,32333,40521,2013,Moto,LAMBRETTA,LI,150.0,NaN,Benc,Aut,0,RIO IBAÑEZ,2025-03-31T00:00:00
6,7,2026-03-06T00:00:00,2026,PRESENCIAL,Total,MT139000113,RENCA,RIO IBANEZ,33715,37283,2013,Moto,LAMBRETTA,LI,150.0,NaN,Benc,Aut,0,RIO IBAÑEZ,2026-03-31T00:00:00
7,8,2026-03-06T00:00:00,2026,PRESENCIAL,Total,MT139000113,RENCA,RIO IBANEZ,34876,34876,2013,Moto,LAMBRETTA,LI,150.0,NaN,Benc,Aut,0,RIO IBAÑEZ,2027-03-31T00:00:00
8,9,2026-03-25T00:00:00,2026,PRESENCIAL,Total,CT166004407,COYHAIQUE,RIO IBANEZ,34876,34876,2007,CAMIONETA,MITSUBISHI,L200,2500.0,Full,Dies,Mec,0,COYHAIQUE,2027-03-31T00:00:00
9,10,2026-04-14T00:00:00,2026,ONLINE,Total,MT032005815,LO BARNECHEA,RIO IBANEZ,34876,34977,2015,MOTO1,B.M.W,G,650.0,Norm,Benc,Mec,0,RIO IBAÑEZ,2027-03-31T00:00:00
